In [1]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import pad_sequences

# Load and preprocess reviews
df = pd.read_csv("LP IV Updated Datasets/CBOW/IMDB Dataset.csv").dropna()
sentences = []
for review in df.review:
    words = preprocess_string(review)
    sentences.append(words)
y = LabelEncoder().fit_transform(df.sentiment)
train, test, y_train, y_test = train_test_split(sentences, y, test_size=0.2)

# Train CBOW on training words before adding padding
cbow = Word2Vec(train, min_count=1, sg=0, epochs=20)

# Give every review 300 token positions; empty padding is ignored when averaging
train = pad_sequences(
    train, maxlen=300, dtype=object, value="", padding="post", truncating="post"
)
test = pad_sequences(
    test, maxlen=300, dtype=object, value="", padding="post", truncating="post"
)

# Average word vectors for each review
x_train = []
for words in train:
    vector = cbow.wv.get_mean_vector(words)
    x_train.append(vector)
x_train = np.array(x_train)

x_test = []
for words in test:
    vector = cbow.wv.get_mean_vector(words)
    x_test.append(vector)
x_test = np.array(x_test)

# Train sentiment classifier
model = Sequential([
    Input(shape=(cbow.vector_size,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=20, batch_size=128)

loss, accuracy = model.evaluate(x_test, y_test)
print("Accuracy:", accuracy * 100, "%")


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8083 - loss: 0.4900
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8602 - loss: 0.3373
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8654 - loss: 0.3192
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8670 - loss: 0.3152
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8673 - loss: 0.3130
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8679 - loss: 0.3117
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8683 - loss: 0.3101
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8696 - loss: 0.3093
Epoch 9/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8699 - loss: 0.3079
Epoch 10/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8703 - loss: 0.3067
Epoch 11/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8698 - loss: 0.3060
Epoch 12/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step